# 🔗 10.16 Joins in Spark

Welcome to **SECTION E: COMBINING DATA**!

Up until now, we have only been working with *one* DataFrame at a time. We cleaned it, formatted it, and aggregated it.

But in the real world, data is rarely stored in one giant table. A company will usually have an `Employees` database and a separate `Departments` database. If the CEO asks you for a report showing the name of every employee alongside the name of their department, you have to stitch those two tables together.

In this lesson, we will learn how to combine datasets across a massive distributed cluster using **Joins**. If you know SQL, the logic will be very familiar, but we will explore how to write them cleanly in PySpark.

### 🎯 Learning Objectives
* Understand the concept of linking tables using a common "Key".
* Master the syntax for the five core Join types: **Inner, Left, Right, Full, and Cross**.
* Understand how missing data (`nulls`) are generated during Outer Joins.
* Recognize the extreme danger of executing a Cross Join on Big Data.
* Understand the performance implications (Shuffles) of distributed joins.

In [ ]:
# 0. Setup: Let's start our SparkSession and create two DataFrames to join.
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Spark_Joins").master("local[*]").getOrCreate()

# Table 1: Employees (Notice Alice has no department_id, she is a new hire!)
emp_data = [
    (1, "Alice", None), 
    (2, "Bob", 101),
    (3, "Charlie", 102),
    (4, "Diana", 102)
]
emp_df = spark.createDataFrame(emp_data, ["emp_id", "emp_name", "dept_id"])

# Table 2: Departments (Notice dept 103 'HR' has no employees yet!)
dept_data = [
    (101, "Sales"),
    (102, "Engineering"),
    (103, "HR")
]
dept_df = spark.createDataFrame(dept_data, ["dept_id", "dept_name"])

print("--- Employee Table (Left Table) ---")
emp_df.show()

print("--- Department Table (Right Table) ---")
dept_df.show()

## 1. The Concept of Keys and Venn Diagrams

To join two tables, they must have a column in common. We call this the **Join Key**. In our example, both tables share `dept_id`.

When explaining joins, Data Engineers use Venn Diagrams. 
* The **Left Circle** represents the first table you write (Employees).
* The **Right Circle** represents the second table (Departments).

<img src="../assets/Module_10/10_16_01.png" alt="Venn diagrams showing Inner Join (intersection), Left Join (entire left circle), Right Join (entire right circle), and Full Join (both circles completely filled in)." width="75%">

## 2. Inner Join (The Intersection)

An **Inner Join** only keeps rows where the Join Key exists in **BOTH** tables. It is the overlapping intersection of the Venn Diagram.

* **Syntax:** `left_df.join(right_df, "join_key", "how")`
* *Note: If both tables have the exact same column name for the key (like `dept_id`), you can just pass the string `"dept_id"` to avoid duplicate columns!*

In [ ]:
print("--- INNER JOIN ---")
# Keep only employees who have a matching department, AND departments that have a matching employee.

inner_join_df = emp_df.join(dept_df, on="dept_id", how="inner")
inner_join_df.show()

# Notice:
# - Alice is GONE (She had no dept_id)
# - HR is GONE (It had no employees)

## 3. Left Join (Keep Everything on the Left)

A **Left Join** keeps **ALL** rows from the Left Table (Employees), regardless of whether they find a match in the Right Table.
If it doesn't find a match, Spark will fill the missing Right Table columns with `null`.

In [ ]:
print("--- LEFT JOIN ---")
# We want a list of EVERY employee. If they have a department, show it. 
# If they don't, just leave the department blank (null).

left_join_df = emp_df.join(dept_df, on="dept_id", how="left")
left_join_df.show()

# Notice:
# - Alice is BACK! But her dept_name is 'null'.
# - HR is still missing (because it is not in the Left Table).

## 4. Right Join & Full Join

### ➡️ Right Join
A **Right Join** is the exact opposite of a Left Join. It keeps **ALL** rows from the Right Table (Departments). If a department has no employees, the employee columns become `null`.

### 🌕 Full Join (Full Outer Join)
A **Full Join** keeps **EVERYTHING** from both tables. It is the combination of a Left Join and a Right Join. Missing matches on either side become `null`.

In [ ]:
print("--- RIGHT JOIN ---")
right_join_df = emp_df.join(dept_df, on="dept_id", how="right")
right_join_df.show()
# Notice: HR is here, but the employee fields are null!

print("--- FULL JOIN ---")
full_join_df = emp_df.join(dept_df, on="dept_id", how="full")
full_join_df.show()
# Notice: Everyone is here! Alice has a null department, and HR has null employees.

## 5. Cross Join (The Danger Zone ⚠️)

A **Cross Join** (also called a Cartesian Product) does not use a Join Key. 

Instead, it takes *every single row* in the Left Table and matches it with *every single row* in the Right Table.

**Why is it dangerous?**
If you have 1,000 Employees and 1,000 Departments, an Inner Join will likely return ~1,000 rows. A Cross Join will return **1,000,000 rows** ($1000 \times 1000$). 

If you do a Cross Join on two Petabyte-scale tables, your cluster will instantly crash with an Out-Of-Memory error.

In [ ]:
print("--- CROSS JOIN ---")
# Because Cross Joins are so dangerous, Spark forces you to explicitly use a different method 
# so you don't do it by accident!

cross_join_df = emp_df.crossJoin(dept_df)
cross_join_df.show()

print(f"Rows in Employees: {emp_df.count()}")
print(f"Rows in Departments: {dept_df.count()}")
print(f"Rows after Cross Join: {cross_join_df.count()} (4 x 3 = 12)")

## 🧑‍💻 The Modern Data Ecosystem: Role Comparison

Joins are the most expensive operation in Data Engineering. Here is how different roles view them:

| Feature | 🏛️ Traditional DBA | 🛠️ Data Engineer (You) | 🧠 Data Scientist |
| :--- | :--- | :--- | :--- |
| **Implementation** | Writes SQL `JOIN`. Relies on the database engine and Foreign Key indexes to make it fast. | **Writes PySpark `.join()`. Must physically manage how the data moves across the cluster.** | Prefers to work with one massive, pre-joined table (a denormalized 'Feature Store'). |
| **The Bottleneck** | Disk I/O (reading from hard drives). | **The Network Shuffle.** To join data, Spark must send all matching keys to the exact same Executor over the network. | Laptop memory limits. |
| **Optimization** | Creates a B-Tree Index on the Join Key. | **Checks for Data Skew. (e.g., If joining on 'City', and New York has 10 million rows while Austin has 10, the New York node will crash).** | Samples the data if the join is too large. |

---

### 📈 Career Progression Roadmap

1. **Junior DE:** Writes basic `.join()` commands. Doesn't realize that joining two massive tables triggers a massive Network Shuffle that slows the entire cluster to a crawl.
2. **Mid-Level DE (Your Current Phase):** Understands the difference between Inner, Left, and Full joins. Knows to avoid Cross Joins at all costs. Starts looking at the DAG execution plan to see the `Exchange` (Shuffle) steps caused by the join.
3. **Senior DE:** Masters **Broadcast Joins**. A Senior DE knows that if you are joining a massive 1-Terabyte table to a tiny 10-Megabyte lookup table, you can tell Spark to *Broadcast* (copy) the tiny table to every single worker node, completely bypassing the network shuffle!

### 🛠️ Where Data Engineering Fits in Modern Organizations
Data usually starts fully separated (normalized) in backend application databases. The Data Engineer's job is to extract these separate tables, use **Joins** to combine them into wide, easy-to-read tables, and load them into a Data Warehouse for business analysts.

## 🔑 Key Takeaways

- **Joins** combine two DataFrames based on a shared "Join Key".
- **Inner Join:** Keeps only the overlapping data (must exist in both tables).
- **Left Join:** Keeps everything from the first (Left) table, filling missing matches with `null`.
- **Full Join:** Keeps everything from both tables.
- **Cross Join:** Matches every row to every row. Extremely dangerous on Big Data ($N \times M$ rows).
- **The Cost:** Joins are **Wide Transformations**. They trigger a massive Network Shuffle to bring matching keys to the same worker node.

## 📝 Knowledge Check Quiz

**Question 1:** You are joining a `Customers` table (Left) to an `Orders` table (Right). You want a list of EVERY customer, even if they have never placed an order. Which join type should you use?
A) Inner Join
B) Right Join
C) Left Join
D) Cross Join

**Question 2:** Why are Joins considered highly expensive operations in Apache Spark?
A) Because they require writing code in Java.
B) Because Joins are "Wide Transformations." Spark must execute a Network Shuffle to physically move data across the cluster so that matching keys end up on the same Executor.
C) Because they delete data.
D) Because Joins require human approval before running.

**Question 3:** What is the primary danger of accidentally executing a Cross Join (Cartesian Product) on two large DataFrames?
A) It deletes the Join Keys.
B) It drops all `null` values.
C) It multiplies the row counts (e.g., 10,000 rows x 10,000 rows = 100,000,000 rows), causing the data volume to explode and likely crashing the cluster with an Out of Memory error.
D) It converts the data to a Python list.

---

*Answers: 1: C, 2: B, 3: C*

In [ ]:
# Clean up our session
spark.stop()
print("Spark session closed.")

### 🚀 What's Next?
You now know how to combine tables *horizontally* (adding columns side-by-side using Joins). 

But what if you want to combine tables *vertically* (stacking rows on top of each other, like combining January's sales data with February's sales data)? 

In the next lesson, **10.17 Union & Deduplication**, we will learn how to stack DataFrames and remove duplicates. Let's go!